In [7]:
%pip install -q torchmetrics
%pip install -q ultralytics

In [8]:
from ultralytics import YOLO
from torch.utils.data import DataLoader, TensorDataset, random_split
from torchvision import datasets, transforms
from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from pathlib import Path
from datasets import load_dataset
import csv
import yaml
import time
import pandas as pd
from torchmetrics.classification import MulticlassJaccardIndex, MulticlassConfusionMatrix, MulticlassF1Score
import matplotlib.pyplot as plt
import cv2
import shutil

In [9]:
SEED = 51
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 8
torch.manual_seed(SEED)
DEVICE

'cuda'

In [10]:
DATASET_ID = "nikolkoo/SatelliteSegmentation"
ROOT = Path.home() / "data" / "SatelliteSegmentation"
IMAGE_DIR = ROOT / "images"
MASK_DIR = ROOT / "masks"
IMAGE_DIR.mkdir(parents=True, exist_ok=True)
MASK_DIR.mkdir(parents=True, exist_ok=True)
OUT_ROOT = Path.home() / "data" / "SatelliteSegmentation" / "filtered"
OUT_IMAGE_DIR = OUT_ROOT / "images"
OUT_MASK_DIR = OUT_ROOT / "masks"
OUT_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
OUT_MASK_DIR.mkdir(parents=True, exist_ok=True)


# Descargar dataset

print("Conectando con Hugging Face")
ds = load_dataset(DATASET_ID, split="train")
print(f"\nImágenes encontradas: {len(ds)}")
print(f"Columnas: {ds.column_names}")


# Descargar imágenes y máscaras
manifest = []
for i, sample in enumerate(ds):
    image = sample["image"]
    mask = sample["mask"]

    image_name = f"{i:04d}.png"
    mask_name = f"{i:04d}_mask.png"

    image_path = IMAGE_DIR / image_name
    mask_path = MASK_DIR / mask_name
    image.convert("RGB").save(image_path)

    # Conservar los valores originales de las etiquetas
    mask.save(mask_path)

    manifest.append({
        "id": i,
        "image": image_name,
        "mask": mask_name,
        "width": image.width,
        "height": image.height,
        "mask_mode": mask.mode
    })
    if (i + 1) % 100 == 0:
        print(f"Descargadas: {i + 1}/{len(ds)}")


# Guardar manifiesto
with open(ROOT / "manifest.csv", "w", newline="") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=manifest[0].keys()
    )
    writer.writeheader()
    writer.writerows(manifest)


print("\nDescarga completada.")
print(f"Imágenes: {len(list(IMAGE_DIR.glob('*.png')))}")
print(f"Máscaras: {len(list(MASK_DIR.glob('*.png')))}")
print(f"Directorio: {ROOT}")

Conectando con Hugging Face

Imágenes encontradas: 790
Columnas: ['image', 'mask']
Descargadas: 100/790
Descargadas: 200/790
Descargadas: 300/790
Descargadas: 400/790
Descargadas: 500/790
Descargadas: 600/790
Descargadas: 700/790

Descarga completada.
Imágenes: 790
Máscaras: 790
Directorio: /root/data/SatelliteSegmentation


In [11]:
MASK_DIR = Path.home() / "data" / "SatelliteSegmentation" / "masks"
IMAGE_DIR = Path.home() / "data" / "SatelliteSegmentation" / "images"

print("Imágenes encontradas:", len(list(IMAGE_DIR.glob("*.png"))))
print("Máscaras encontradas:", len(list(MASK_DIR.glob("*.png"))))

Imágenes encontradas: 790
Máscaras encontradas: 790


In [12]:
class_names = {
    0: "Sin etiqueta",
    10: "Árboles",
    20: "Matorrales",
    30: "Pastizales",
    40: "Cultivos",
    50: "Zonas construidas",
    60: "Suelo desnudo / vegetación escasa",
    70: "Nieve y hielo",
    80: "Cuerpos de agua",
    90: "Humedales",
    95: "Manglares",
    100: "Musgos y líquenes"
}

CLASES_A_ELIMINAR = {0, 20, 70, 100}

label_mapping = {
    10: 0, 
    30: 1,  
    40: 2,  
    50: 3,  
    60: 4,  
    80: 5,  
    90: 6, 
    95: 7   
}

model_class_names = {
    new_id: class_names[orig_id]
    for orig_id, new_id in label_mapping.items()
}

# 5. Emparejar archivos siguiendo tu estructura
image_files = sorted(IMAGE_DIR.glob("*.png"))
mask_files = sorted(MASK_DIR.glob("*_mask.png"))

images_by_name = {f.stem: f for f in image_files}
masks_by_name = {f.stem.replace("_mask", ""): f for f in mask_files}

common_names = sorted(set(images_by_name) & set(masks_by_name))

print(f"Imágenes totales: {len(image_files)} | Máscaras totales: {len(mask_files)}")
print(f"Pares válidos encontrados: {len(common_names)}\n")

# 6. Filtrar y re-indexar
conservados = 0
eliminados = 0

for name in common_names:
    img_path = images_by_name[name]
    mask_path = masks_by_name[name]
    
    mask = np.array(Image.open(mask_path))
    clases_presentes = set(np.unique(mask))
    
    # Verificar si contiene al menos un píxel de las clases a eliminar
    if clases_presentes & CLASES_A_ELIMINAR:
        eliminados += 1
        continue
        
    # Re-indexar la máscara usando los nuevos IDs consecutivos (0-7)
    nueva_mask = np.zeros_like(mask, dtype=np.uint8)
    for orig_id, new_id in label_mapping.items():
        nueva_mask[mask == orig_id] = new_id
        
    # Guardar máscara re-indexada y copiar imagen original
    Image.fromarray(nueva_mask).save(OUT_MASK_DIR / mask_path.name)
    shutil.copy2(img_path, OUT_IMAGE_DIR / img_path.name)
    conservados += 1

print("--- Resumen del filtrado ---")
print(f"Pares conservados: {conservados}")
print(f"Pares eliminados:  {eliminados}")
print("\nNuevo diccionario de clases para tu modelo:")
for k, v in model_class_names.items():
    print(f"  {k}: {v}")

Imágenes totales: 790 | Máscaras totales: 790
Pares válidos encontrados: 790

--- Resumen del filtrado ---
Pares conservados: 701
Pares eliminados:  89

Nuevo diccionario de clases para tu modelo:
  0: Árboles
  1: Pastizales
  2: Cultivos
  3: Zonas construidas
  4: Suelo desnudo / vegetación escasa
  5: Cuerpos de agua
  6: Humedales
  7: Manglares
